<a href="https://www.kaggle.com/code/manhhungtr211/inference-trm-with-finance-dataset?scriptVersionId=322490470" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## Tiny Recursive Model for Text Generation
Tiny Recursive Model (TRM) for Text Generation

Based on: "Less is More: Recursive Reasoning with Tiny Networks" by A. Jolicoeur-Martineau (2025)

This implementation adapts TRM for autoregressive text generation:
- Uses recursive reasoning with a tiny 2-layer transformer
- Deep supervision with latent state carried across improvement steps
- Single network architecture (no hierarchical split)
- EMA for training stability

Key insight from the paper: smaller networks with deep recursion can outperform
larger networks by avoiding overfitting while achieving high effective depth.

## Note
- inference trước
- xài token llama -> ok
- tăng tốc độ học lr = 10^-3 -> -2 -> ok

## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
import os
from tqdm import tqdm
import copy
from torch.utils.checkpoint import checkpoint
from transformers import AutoTokenizer


## Building blocks
for the base transformer, including rotary embeddings, RMS normalization, SwiGLU activation

In [2]:
# ============================================================================
# Model Architecture
# ============================================================================

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization"""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        return x / rms * self.weight


class RotaryEmbedding(nn.Module):
    """Rotary Position Embedding (RoPE)"""
    def __init__(self, dim, max_seq_len=512):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
        self.max_seq_len = max_seq_len
        self._build_cache(max_seq_len)

    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, device=self.inv_freq.device)
        freqs = torch.einsum('i,j->ij', t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer('cos_cached', emb.cos())
        self.register_buffer('sin_cached', emb.sin())

    def forward(self, x):
        seq_len = x.shape[1]
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin):
    # Original cos/sin shape: [seq_len, head_dim]
    # q/k shape: [batch_size, n_heads, seq_len, head_dim]
    # We need cos/sin to be [1, 1, seq_len, head_dim] for proper broadcasting
    cos = cos.unsqueeze(0).unsqueeze(1)  # Corrected from unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(1)  # Corrected from unsqueeze(2)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class SwiGLU(nn.Module):
    """SwiGLU activation function"""
    def __init__(self, dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, dim, bias=False)
        self.w3 = nn.Linear(dim, hidden_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with RoPE"""
    def __init__(self, dim, n_heads, max_seq_len=512):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads

        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        self.rope = RotaryEmbedding(self.head_dim, max_seq_len)

        # Causal mask
        mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer('mask', mask)

    def forward(self, x):
        B, T, C = x.shape

        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        cos, sin = self.rope(x)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)

        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.mask[:T, :T], float('-inf'))
        att = F.softmax(att, dim=-1)

        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)


class TransformerBlock(nn.Module):
    """Single transformer block with pre-norm"""
    def __init__(self, dim, n_heads, mlp_ratio=4, max_seq_len=512):
        super().__init__()
        self.norm1 = RMSNorm(dim)
        self.attn = CausalSelfAttention(dim, n_heads, max_seq_len)
        self.norm2 = RMSNorm(dim)
        self.mlp = SwiGLU(dim, dim * mlp_ratio)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


## The Tiny Recursive Model

In [3]:

class TinyRecursiveNetwork(nn.Module):
    """
    The core tiny network used in TRM.
    Only 2 layers as per the paper's finding that smaller is better.
    """
    def __init__(self, dim, n_heads=8, n_layers=2, mlp_ratio=4, max_seq_len=512):
        super().__init__()
        self.layers = nn.ModuleList([
            TransformerBlock(dim, n_heads, mlp_ratio, max_seq_len)
            for _ in range(n_layers)
        ])
        self.norm = RMSNorm(dim)

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)


class TinyRecursiveModel(nn.Module):
    """
    Tiny Recursive Model for Text Generation

    Architecture based on TRM paper:
    - Single tiny 2-layer network
    - Recursive reasoning with latent z and prediction y
    - Deep supervision across multiple improvement steps

    For text generation:
    - x: embedded input sequence (context)
    - y: current token predictions (embedded)
    - z: latent reasoning state

    The model recursively improves its latent z, then updates y.
    """
    def __init__(
        self,
        vocab_size,
        dim=256,
        n_heads=8,
        n_layers=2,
        mlp_ratio=4,
        max_seq_len=256,
        n_latent_recursions=6,  # n in the paper
        n_improvement_cycles=3,  # T in the paper
    ):
        super().__init__()
        self.dim = dim
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.n_latent_recursions = n_latent_recursions
        self.n_improvement_cycles = n_improvement_cycles

        # Embeddings
        self.token_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)

        # Single tiny network (key insight: one network is better than two)
        self.net = TinyRecursiveNetwork(dim, n_heads, n_layers, mlp_ratio, max_seq_len)

        # Projection layers for combining x, y, z
        self.combine_xyz = nn.Linear(dim * 3, dim, bias=False)
        self.combine_yz = nn.Linear(dim * 2, dim, bias=False)

        # Output head
        self.output_head = nn.Linear(dim, vocab_size, bias=False)

        # Halting head for ACT (simplified - no Q-learning)
        self.halt_head = nn.Linear(dim, 1, bias=False)

        # Learnable initial states for y and z
        self.y_init = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.z_init = nn.Parameter(torch.randn(1, 1, dim) * 0.02)

        self._init_weights()

    def _init_weights(self):
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def get_embeddings(self, input_ids):
        """Get token + position embeddings"""
        B, T = input_ids.shape
        # Clamp input_ids to valid range
        input_ids = input_ids.clamp(0, self.vocab_size - 1)
        # Clamp position to max_seq_len
        T = min(T, self.max_seq_len)
        pos = torch.arange(T, device=input_ids.device).unsqueeze(0)
        return self.token_emb(input_ids[:, :T]) + self.pos_emb(pos)

    def latent_recursion(self, x, y, z):
        """
        Single recursion cycle:
        1. Update z n times given (x, y, z)
        2. Update y once given (y, z)
        """
        # Latent reasoning: update z n times
        for _ in range(self.n_latent_recursions):
            combined = self.combine_xyz(torch.cat([x, y, z], dim=-1))
            z = self.net(combined)

        # Refine prediction: update y given (y, z)
        combined_yz = self.combine_yz(torch.cat([y, z], dim=-1))
        y = self.net(combined_yz)

        return y, z

    def deep_recursion(self, x, y, z, use_grad=True):
        """
        Deep recursion with T improvement cycles.
        First T-1 cycles without gradients, last cycle with gradients.
        """
        if not use_grad:
            # All cycles without gradients (inference)
            with torch.no_grad():
                for _ in range(self.n_improvement_cycles):
                    y, z = checkpoint(self.latent_recursion, x, y, z, use_reentrant=False)
            return y.detach(), z.detach()

        # T-1 cycles without gradients
        with torch.no_grad():
            for _ in range(self.n_improvement_cycles - 1):
                y, z = self.latent_recursion(x, y, z)

        # Last cycle with gradients
        y, z = self.latent_recursion(x, y, z)

        return y.detach(), z.detach(), self.output_head(y), self.halt_head(y.mean(dim=1))

    def forward(self, input_ids, targets=None, n_supervision_steps=4):
        """
        Forward pass with deep supervision.

        Args:
            input_ids: [B, T] input token IDs
            targets: [B, T] target token IDs (for training)
            n_supervision_steps: number of deep supervision steps

        Returns:
            If training: loss
            If inference: logits
        """
        B, T = input_ids.shape
        T = min(T, self.max_seq_len)
        input_ids = input_ids[:, :T]

        x = self.get_embeddings(input_ids)

        # Initialize y and z
        y = self.y_init.expand(B, T, -1).clone()
        z = self.z_init.expand(B, T, -1).clone()

        if targets is None:
            # Inference: just run deep recursion
            y, z = self.deep_recursion(x, y, z, use_grad=False)
            return self.output_head(y)

        # Ensure targets match input length
        targets = targets[:, :T]

        # Training with deep supervision
        total_loss = 0.0

        for step in range(n_supervision_steps):
            y, z, logits, halt_logit = self.deep_recursion(x, y, z, use_grad=True)

            # Cross-entropy loss for token prediction
            ce_loss = F.cross_entropy(
                logits.view(-1, self.vocab_size),
                targets.reshape(-1),
                ignore_index=-100
            )

            # Halting loss (simplified ACT)
            with torch.no_grad():
                preds = logits.argmax(dim=-1)
                mask = (targets != -100)
                correct = ((preds == targets) & mask).float().sum() / mask.float().sum().clamp(min=1)
            halt_loss = F.binary_cross_entropy_with_logits(
                halt_logit.squeeze(-1),
                correct.expand(B)
            )

            total_loss = total_loss + ce_loss + 0.1 * halt_loss

        return total_loss / n_supervision_steps

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=0.8, top_k=40):
        """Generate text autoregressively"""
        self.eval()

        for _ in range(max_new_tokens):
            # Crop to max_seq_len - 1 to leave room for prediction
            idx_cond = input_ids[:, -(self.max_seq_len - 1):]

            # Clamp input ids to valid vocab range
            idx_cond = idx_cond.clamp(0, self.vocab_size - 1)

            # Get predictions
            logits = self(idx_cond)
            logits = logits[:, -1, :] / temperature

            # Top-k sampling
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids



In [4]:

# ============================================================================
# Dataset
# ============================================================================

class FinanceTasksDataset(Dataset):
    """Dataset for AdaptLLM/finance-tasks ConvFinQA"""
    def __init__(self, tokenizer, split='train', max_length=256, max_samples=None):
        print(f"Loading AdaptLLM/finance-tasks ConvFinQA {split} split...")
        # ConvFinQA only has test split, so we load test and split manually if needed
        dataset = load_dataset('AdaptLLM/finance-tasks', 'ConvFinQA', split='test')
        
        # create a pseudo train/val split using a fixed seed
        dataset = dataset.train_test_split(test_size=0.1, seed=42)
        dataset = dataset['train'] if split == 'train' else dataset['test']
        
        if max_samples:
            dataset = dataset.select(range(min(max_samples, len(dataset))))

        self.tokenizer = tokenizer
        self.max_length = max_length
        
        # Format ConvFinQA data: input + '\n\n' + label
        self.texts = [row['input'] + "\n\n" + str(row['label']) for row in dataset]
        self.vocab_size = tokenizer.vocab_size
        self.pad_token_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
        print(f"Loaded {len(self.texts)} samples")

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        # Add BOS/EOS handling
        tokens = self.tokenizer.encode(text, truncation=True, max_length=self.max_length)

        # Ensure all tokens are within valid range
        tokens = [min(max(t, 0), self.vocab_size - 1) for t in tokens]

        # Pad if necessary
        if len(tokens) < self.max_length:
            tokens = tokens + [self.pad_token_id] * (self.max_length - len(tokens))
        else:
            tokens = tokens[:self.max_length]

        tokens = torch.tensor(tokens, dtype=torch.long)

        # Input is tokens[:-1], target is tokens[1:]
        input_ids = tokens[:-1].clone()
        targets = tokens[1:].clone()

        # Mask padding in targets (set to -100 to ignore in loss)
        targets[targets == self.pad_token_id] = -100

        return input_ids, targets


## Exponential Moving Average
for weight regularization

In [5]:
class EMA:
    """Exponential Moving Average for model weights"""
    def __init__(self, model, decay=0.999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}

        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (
                    self.decay * self.shadow[name] +
                    (1 - self.decay) * param.data
                )

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]


## Main Training Function

In [6]:
def train(
    model,
    train_loader,
    val_loader,
    tokenizer,
    device,
    epochs=5,
    lr=1e-3,
    warmup_steps=1000,
    n_supervision_steps=4,
    ema_decay=0.999,
    save_path='trm_finance.pt'
):
    """Training loop with deep supervision and EMA"""

    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.95), weight_decay=0.1)
    ema = EMA(model, decay=ema_decay)

    # Learning rate scheduler with warmup
    def lr_schedule(step):
        if step < warmup_steps:
            return step / warmup_steps
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_schedule)

    global_step = 0
    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs}')

        for input_ids, targets in pbar:
            input_ids = input_ids.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            loss = model(input_ids, targets, n_supervision_steps=n_supervision_steps)
            
            # -------------------------------------------------------------
            # 🔥 FIX DUAL GPU CHO TRAIN LOOP: Gom loss trước khi backward
            if loss.dim() > 0:
                loss = loss.mean()
            # -------------------------------------------------------------
            
            loss.backward()

            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()
            ema.update()

            global_step += 1
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'lr': f'{scheduler.get_last_lr()[0]:.6f}'})

        # Validation
        ema.apply_shadow()
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for input_ids, targets in tqdm(val_loader, desc='Validation'):
                input_ids = input_ids.to(device)
                targets = targets.to(device)
                loss = model(input_ids, targets, n_supervision_steps=n_supervision_steps)
                
                # -------------------------------------------------------------
                # 🔥 FIX DUAL GPU CHO VAL LOOP: Gom loss trước khi gọi .item()
                if loss.dim() > 0:
                    loss = loss.mean()
                # -------------------------------------------------------------
                
                val_loss += loss.item()

        val_loss /= len(val_loader)
        print(f'Epoch {epoch+1} - Val Loss: {val_loss:.4f}')

        # Generate sample
        # Lấy mẫu số 0 từ tập validation để test khả năng trả lời QA thực tế
        eval_text = val_loader.dataset.texts[0]
        eval_input = eval_text.rsplit("\n\n", 1)[0] + "\n\n"
        eval_label = eval_text.rsplit("\n\n", 1)[1]
        
        prompt_tokens = tokenizer.encode(eval_input)
        
        # Cắt bớt phần đầu context nếu quá dài, giữ lại câu hỏi ở đuôi
        # Lưu ý: Chắc chắn rằng model.module.max_seq_len được gọi nếu dùng DataParallel
        max_seq_len = model.module.max_seq_len if hasattr(model, 'module') else model.max_seq_len
        
        if len(prompt_tokens) > max_seq_len - 1:
            prompt_tokens = prompt_tokens[-(max_seq_len - 1):]
            
        prompt_ids = torch.tensor([prompt_tokens], device=device)
        
        # Dùng temp thấp để mô hình trả lời số liệu logic (tránh sinh bừa bãi)
        # Sửa lại lệnh generate để tương thích với DataParallel
        model_for_gen = model.module if hasattr(model, 'module') else model
        generated = model_for_gen.generate(prompt_ids, max_new_tokens=15, temperature=0.1) 
        
        # Chỉ lấy phần câu trả lời mà model sinh ra (bỏ phần câu hỏi mồi)
        generated_text = tokenizer.decode(generated[0][prompt_ids.shape[1]:].tolist())
        
        print(f'\n[Test QA]')
        print(f'- Question Snippet: ...{eval_input[-100:]}'.replace('\n', ' '))
        print(f'- Generated Answer: {generated_text.strip()}')
        print(f'- True Answer:      {eval_label}\n')

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            
            # 🔥 Sửa lại cách lưu weights chuẩn cho DataParallel
            model_state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            
            torch.save({
                'model_state_dict': model_state,
                'ema_shadow': ema.shadow,
                'epoch': epoch,
                'val_loss': val_loss
            }, save_path)
            print(f'Saved best model with val_loss={val_loss:.4f}')

        ema.restore()

    return model

## Configuration

In [7]:
# ============================================================================
# Main
# ============================================================================

# def main():
# Configuration
config = {
    'vocab_size': 50257,       # GPT-2 vocab
    'dim': 256,                # Hidden dimension
    'n_heads': 8,              # Attention heads
    'n_layers': 4,             # Only 2 layers (key insight from paper)
    'mlp_ratio': 16,
    'max_seq_len': 128,        # Reduced for stability
    'n_latent_recursions': 4,  # n in paper (reduced for memory)
    'n_improvement_cycles': 2, # T in paper (reduced for memory)

    # Training
    'batch_size': 16,          # ĐÃ SỬA: Giảm từ 256 xuống 16 để tránh lỗi OOM
    'epochs': 3,
    'lr': 1e-4,
    'warmup_steps': 500,
    'n_supervision_steps': 3,  # Deep supervision steps during training
    'max_train_samples': 2000000,  # Limit for faster training demo
    'max_val_samples': 20000,
}

## Instantiate Model

In [8]:

# Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

# Model
model = TinyRecursiveModel(
    vocab_size=config['vocab_size'],
    dim=config['dim'],
    n_heads=config['n_heads'],
    n_layers=config['n_layers'],
    mlp_ratio=config['mlp_ratio'],
    max_seq_len=config['max_seq_len'],
    n_latent_recursions=config['n_latent_recursions'],
    n_improvement_cycles=config['n_improvement_cycles'],
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. KÍCH HOẠT DUAL GPU 🚀
if torch.cuda.device_count() > 1:
    print(f"🔥 Kích hoạt thành công {torch.cuda.device_count()} GPUs!")
    # DataParallel sẽ tự động chia nhỏ Batch Size ra cho các GPU
    model = nn.DataParallel(model)

# 4. Đẩy model vào thiết bị
model = model.to(device)

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,} ({n_params/1e6:.2f}M)')
print(f'Effective depth per supervision step: {config["n_improvement_cycles"] * (config["n_latent_recursions"] + 1) * config["n_layers"]}')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

🔥 Kích hoạt thành công 2 GPUs!
Model parameters: 39,726,592 (39.73M)
Effective depth per supervision step: 40


## Load dataset

In [9]:
# Datasets
train_dataset = FinanceTasksDataset(
    tokenizer,
    split='train',
    max_length=config['max_seq_len'] + 1,  # +1 for next token prediction
    max_samples=config['max_train_samples']
)
val_dataset = FinanceTasksDataset(
    tokenizer,
    split='validation',
    max_length=config['max_seq_len'] + 1,
    max_samples=config['max_val_samples']
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config['batch_size'],
    shuffle=True,
    num_workers=2,
    pin_memory=True
)
val_loader = DataLoader(
    val_dataset,
    batch_size=config['batch_size'],
    num_workers=2,
    pin_memory=True
)

Loading AdaptLLM/finance-tasks ConvFinQA train split...


README.md: 0.00B [00:00, ?B/s]

ConviFinQA/test.json:   0%|          | 0.00/5.82M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1490 [00:00<?, ? examples/s]

Loaded 1341 samples
Loading AdaptLLM/finance-tasks ConvFinQA validation split...
Loaded 149 samples


In [10]:
for i, (input_ids, targets) in enumerate(train_loader):
    
    # Forward pass (trả về 1 vector chứa loss của các GPUs)
    loss = model(input_ids, targets, n_supervision_steps=config['n_supervision_steps'])
    
    # -----------------------------------------
    # 🔥 DÒNG FIX LỖI: Gom loss của các GPU lại thành 1 số duy nhất
    if loss.dim() > 0:
        loss = loss.mean()
    # -----------------------------------------
    
    # Bây giờ loss đã là 1 số vô hướng, backward sẽ chạy mượt mà!
    loss.backward()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


## Train!

In [11]:
# Train
model = train(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    tokenizer=tokenizer,
    device=device,
    epochs=config['epochs'],
    lr=config['lr'],
    warmup_steps=config['warmup_steps'],
    n_supervision_steps=config['n_supervision_steps'],
)

print('\nTraining complete!')


Validation: 100%|██████████| 10/10 [00:04<00:00,  2.41it/s]
Token indices sequence length is longer than the specified maximum sequence length for this model (1172 > 1024). Running this sequence through the model will result in indexing errors


Epoch 1 - Val Loss: 10.9287

[Test QA]
- Question Snippet: ... 1432.0  and how much does this change represent in relation to those fees in 2005, in percentage?  
- Generated Answer: stains edges Protective stainsLen 112 classmates 112 keeps 112 stacking 112 rates publications accounted
- True Answer:      0.35029

Saved best model with val_loss=10.9287


Validation: 100%|██████████| 10/10 [00:04<00:00,  2.41it/s]


Epoch 2 - Val Loss: 10.8742

[Test QA]
- Question Snippet: ... 1432.0  and how much does this change represent in relation to those fees in 2005, in percentage?  
- Generated Answer: wanting stains 112 stains separGreenexpress stains 112 Bull 112 stains HS stains ducks
- True Answer:      0.35029

Saved best model with val_loss=10.8742


Validation: 100%|██████████| 10/10 [00:04<00:00,  2.41it/s]


Epoch 3 - Val Loss: 10.7605

[Test QA]
- Question Snippet: ... 1432.0  and how much does this change represent in relation to those fees in 2005, in percentage?  
- Generated Answer: suspension 112 ElvisIES Klingon 112 and stains 112 thecair 112 ducks obser 112
- True Answer:      0.35029

Saved best model with val_loss=10.7605

Training complete!


## Inference

In [12]:
!rm -rf /kaggle/working/Adapts_edited

In [13]:
# Di chuyển về thư mục làm việc của Kaggle
%cd /kaggle/working

# Clone repo benchmark-adaptllm
!git clone https://github.com/manhhungtr211/Adapts_edited

# Vào trong repo
%cd Adapts_edited

/kaggle/working
Cloning into 'Adapts_edited'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 185 (delta 68), reused 159 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (185/185), 869.03 KiB | 24.83 MiB/s, done.
Resolving deltas: 100% (68/68), done.
/kaggle/working/Adapts_edited


In [14]:
!pip install -r requirements.txt
!pip install transformers accelerate datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of datasets to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.5/78.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━

In [15]:
import os

# Giữ cache trong /kaggle/temp
os.environ["HF_HOME"] = "/kaggle/temp/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/kaggle/temp/huggingface/cache"
os.environ["HF_HUB_CACHE"] = "/kaggle/temp/huggingface/cache"
os.environ["TMPDIR"] = "/kaggle/temp/huggingface/tmp"

# Tạo thư mục
os.makedirs("/kaggle/temp/huggingface/cache", exist_ok=True)
os.makedirs("/kaggle/temp/huggingface/tmp", exist_ok=True)

# Tạo liên kết trong /kaggle/working để theo dõi
!ln -s /kaggle/temp/huggingface /kaggle/working/huggingface_link

print("📂 Đã tạo liên kết symbolic để theo dõi cache trong sidebar.")
!ls -lh /kaggle/working

📂 Đã tạo liên kết symbolic để theo dõi cache trong sidebar.
total 152M
drwxr-xr-x 9 root root 4.0K May 27 10:29 Adapts_edited
lrwxrwxrwx 1 root root   24 May 27 10:29 huggingface_link -> /kaggle/temp/huggingface
---------- 1 root root 227K May 27 10:29 __notebook__.ipynb
-rw-r--r-- 1 root root 152M May 27 10:29 trm_finance.pt


In [16]:
!python raw2read.py

max_workers: 4
loading raw texts in the input folder...
paths: ['./data_samples/input-raw-texts/0.txt', './data_samples/input-raw-texts/1.txt', './data_samples/input-raw-texts/10.txt', './data_samples/input-raw-texts/11.txt', './data_samples/input-raw-texts/2.txt', './data_samples/input-raw-texts/3.txt', './data_samples/input-raw-texts/4.txt', './data_samples/input-raw-texts/5.txt', './data_samples/input-raw-texts/6.txt', './data_samples/input-raw-texts/7.txt', './data_samples/input-raw-texts/8.txt', './data_samples/input-raw-texts/9.txt']
12it [00:00, 21704.03it/s]
transferring raw texts into reading comprehension...
100%|███████████████████████████████████████████| 12/12 [00:00<00:00, 16.79it/s]
saving reading comprehension texts...
saved to ./data_samples/output-read-compre


In [17]:
import torch
print(torch.cuda.device_count())

2


In [18]:
!python inference.py \
  model_name="gpt2" \
  checkpoint_path="/kaggle/working/trm_finance.pt" \
  task_name="ConvFinQA" \
  add_bos_token=False \
  model_parallel=True

/kaggle/working/Adapts_edited/inference.py:329: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path="configs", config_name="inference")
/usr/local/lib/python3.12/dist-packages/hydra/_internal/hydra.py:119: UserWarning: Future Hydra versions will no longer change working directory at job runtime by default.
See https://hydra.cc/docs/1.2/upgrades/1.1_to_1.2/changes_to_job_working_dir/ for more information.
  ret = run_job(
[2026-05-27 10:29:54,078][__main__][INFO] - {'model_name': 'gpt2', 'task_name': 'ConvFinQA', 'checkpoint_path': '/kaggle/working/trm_finance.pt', 'output_dir': '/tmp/output', 'res_dir': '/tmp/res', 'max_length': 2048, 'generate_max_len': 100, 'n_tokens': 2048, 'cache_dir': '/tmp/cache', 'add_bos_token': False, 'model_parallel': True, 'dataset_reader': {'_target_': 'src.dataset_readers.inference_dsr.InferenceDatasetReader', 'model_name': '${model_